# Handwritten k-nearest neighbours on digits

This notebook demonstrates the reusable API in `mlshowcase/knn.py` and reads the evidence generated by the command-line experiment:

```sh
uv run --locked python -m mlshowcase.knn --output-dir results --seeds 42 43 44
```

The experiment compares a handwritten NumPy k-nearest-neighbour classifier with `sklearn.neighbors.KNeighborsClassifier` on the bundled scikit-learn digits dataset:

- three seeded, stratified 60/20/20 splits (train / dev / test);
- a development sweep over `k = [1, 3, 5, 7, 9, 11, 15]` and three distances: Euclidean, Manhattan and Minkowski with `p = 1.5`;
- selection by development macro F1 with a documented tie-break;
- held-out evaluation only after the configuration is fixed.

**This notebook never retrains the experiment.** It loads [`../results/knn.json`](../results/knn.json) and the generated PNG figures, so *Run All* is cheap, deterministic and offline. The reference notes live in [`../docs/knn.md`](../docs/knn.md).

In [ ]:
import json
import os
import sys
import tomllib
from pathlib import Path

# This notebook runs with its checkout's ``notebooks/`` directory as the working
# directory, so ``..`` is the public checkout root. Set MLSHOWCASE_ROOT to the
# checkout's absolute path to start the notebook from anywhere else.
EXPECTED_PROJECT = "mlshowcase"


def require_showcase_checkout(root: Path, *required_files: str) -> None:
    """Fail before any import or result read unless ``root`` is this checkout."""
    guide = (
        f"{root} is not the mlshowcase checkout this notebook belongs to: run the "
        "notebook with its notebooks/ directory as the working directory, or set "
        "MLSHOWCASE_ROOT to the checkout's absolute path"
    )
    pyproject = root / "pyproject.toml"
    if not pyproject.is_file():
        raise FileNotFoundError(f"{guide} (no pyproject.toml found there)")
    try:
        name = tomllib.loads(pyproject.read_text(encoding="utf-8"))["project"]["name"]
    except (KeyError, tomllib.TOMLDecodeError) as error:
        raise ValueError(f"{guide} (project.name could not be read)") from error
    if name != EXPECTED_PROJECT:
        raise ValueError(f"{guide} (project.name is {name!r})")
    for relative in required_files:
        if not (root / relative).is_file():
            raise FileNotFoundError(f"{guide} (missing {relative})")


_override = os.environ.get("MLSHOWCASE_ROOT")
if _override and not Path(_override).is_absolute():
    raise ValueError(f"MLSHOWCASE_ROOT must be an absolute path, not {_override!r}")
SHOWCASE_ROOT = Path(_override).resolve() if _override else Path("..").resolve()
require_showcase_checkout(SHOWCASE_ROOT, "mlshowcase/knn.py", "notebooks/knn.ipynb")

if str(SHOWCASE_ROOT) not in sys.path:
    sys.path.insert(0, str(SHOWCASE_ROOT))

RESULTS_JSON = SHOWCASE_ROOT / "results" / "knn.json"

In [ ]:
if not RESULTS_JSON.is_file():
    raise FileNotFoundError(
        f"missing {RESULTS_JSON.relative_to(SHOWCASE_ROOT)}; generate the evidence first with "
        "`uv run --locked python -m mlshowcase.knn` from the repository root"
    )

payload = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))
print(f"loaded {RESULTS_JSON.relative_to(SHOWCASE_ROOT)}: {payload['summary']['n_seeds']} seed runs")

## What the evidence file records

Every number below is read from the generated JSON; nothing is recomputed here.

| Field | Contents |
| --- | --- |
| `protocol.provenance` | dataset, loader, sample/feature counts, classes, no-download note |
| `protocol.preprocessing` | the transform applied before fitting (here: none, all 64 features) |
| `protocol.splits` | split strategy, fractions, per-seed split and per-class counts |
| `protocol.seeds`, `protocol.hyperparameters` | seed list, k grid, distance grid, tie-break rules |
| `protocol.selection` | selection criterion, tie-break, held-out discipline |
| `environment` | public-safe Python/OS/CPU/package metadata (no paths or host names) |
| `runs` | per seed: dev curves, selection, held-out accuracy / macro F1 / confusion matrix, timings |
| `summary` | repeat summaries across seeds: means, standard deviations, selection counts |

In [ ]:
protocol = payload["protocol"]
provenance = protocol["provenance"]

print(f"dataset    : {provenance['dataset']}")
print(f"loader     : {provenance['loader']}")
print(f"downloads  : {provenance['network_access_required']}")
print(f"shape      : {provenance['n_samples']} samples x {provenance['n_features']} features")
print(f"classes    : {provenance['classes']}")
print(f"preprocess : {protocol['preprocessing']['steps']} ({protocol['preprocessing']['detail']})")
print(f"seeds      : {protocol['seeds']}")
print(f"k grid     : {protocol['hyperparameters']['k_grid']}")
print(f"distances  : {protocol['hyperparameters']['distances']}")
print(f"selection  : {protocol['selection']['criterion']}")
print(f"tie-break  : {protocol['selection']['tie_break']}")

In [ ]:
print(json.dumps(payload["environment"], indent=2, sort_keys=True))

In [ ]:
for seed, split in protocol["splits"]["per_seed"].items():
    counts = split["counts"]
    print(f"seed {seed}: train={counts['train']} dev={counts['dev']} test={counts['test']}")
    for part in ("train", "dev", "test"):
        class_counts = split["class_counts"][part]
        labels = sorted(class_counts, key=int)
        detail = " ".join(f"{label}:{class_counts[label]}" for label in labels)
        print(f"  {part:5s} {detail}")

## What this rebuild changes from the original coursework

The historical notebook and the notes that accompany it document real problems. This rebuild fixes them instead of copying them, and `tests/test_knn.py` pins each behaviour:

- **Every feature is used.** The original Euclidean and Manhattan loops stopped at `len(row) - 1`, silently dropping the final pixel column, while Minkowski used all components. All three distances here reduce over all 64 features; a test uses a worked example whose answer changes when the last column is dropped.
- **Labels stay separate.** The original appended the label column to the pixel matrix, which turned integer labels into floats, then re-split the combined array. The reusable API keeps features and labels as separate arrays and returns labels with their original integer dtype.
- **Seeded, stratified splits.** The original shuffle was unseeded and the split was not stratified. `stratified_split` reproduces each split from its seed, and the per-class counts are recorded in the evidence file.
- **The sweep is actually executed.** The original tuning loop was commented out and the notebook fixed `k = 5` with Manhattan distance. This evidence comes from a real sweep over three distances and seven values of `k`.
- **Deterministic ties.** Majority-vote ties resolve to the smallest class label and equal distances keep training-row order, so results do not depend on incidental sort behaviour.
- **Held-out discipline.** The test split is scored once per seed, only after the configuration is selected on the development split.
- **A real baseline.** `sklearn.neighbors.KNeighborsClassifier` is fitted with the identical selected hyperparameters on the identical splits, making the comparison an implementation check rather than a tuned competitor.

## Reusable distance API

`euclidean_distance`, `manhattan_distance` and `minkowski_distance` are handwritten with NumPy and reduce over every component of the two feature vectors. Minkowski requires an explicit exponent `p >= 1`; values below 1 are rejected with `ValueError`, because the triangle inequality no longer holds there.

The demonstration below uses hand-checkable vectors: `(3, 4, 12)` against the origin.

In [ ]:
from mlshowcase.knn import (
    KNNClassifier,
    euclidean_distance,
    majority_vote,
    manhattan_distance,
    minkowski_distance,
    rank_neighbors,
)

a = [3.0, 4.0, 12.0]
b = [0.0, 0.0, 0.0]
print("euclidean(a, b)         =", euclidean_distance(a, b), "(sqrt(3^2 + 4^2 + 12^2) = 13)")
print("manhattan(a, b)         =", manhattan_distance(a, b), "(3 + 4 + 12 = 19)")
print("minkowski(a, b, p=1.5)  =", round(minkowski_distance(a, b, p=1.5), 6))

try:
    minkowski_distance(a, b, p=0.5)
except ValueError as error:
    print("p < 1 rejected:", error)

## Reusable classifier API

`KNNClassifier` follows the usual `fit` / `predict` pattern. Fitting stores the training features and labels as separate arrays, so labels keep their own dtype and noncontiguous class values survive the vote unchanged.

Predictions are deterministic: an exact vote tie resolves to the smallest class label, and equal distances keep their training-row order. `rank_neighbors` exposes the neighbour ranking directly, which is what the experiment uses to reuse one distance computation per metric across the whole `k` sweep instead of recomputing distances for every `k`.

In [ ]:
train_x = [[1.0, 0.0], [2.0, 0.0], [3.0, 0.0]]
train_y = [7, 3, 3]  # noncontiguous labels are fine
query = [[0.0, 0.0]]

ranking = rank_neighbors(train_x, query, metric="euclidean")
print("ranked neighbour distances:", ranking.distances[0])
print("vote at k=1:", majority_vote(ranking, train_y, 1)[0])
print("vote at k=3:", majority_vote(ranking, train_y, 3)[0])

model = KNNClassifier(n_neighbors=3).fit(train_x, train_y)
print("classifier prediction (k=3):", model.predict(query)[0])

tie_model = KNNClassifier(n_neighbors=2).fit([[1.0, 0.0], [-1.0, 0.0]], [9, 2])
print("1-1 tie resolved to the smallest label:", tie_model.predict([[0.0, 0.0]])[0])

## Development sweep

The development sweep is recorded per seed: for every distance and every `k` the JSON stores development accuracy and macro F1. The table below averages the macro F1 curves across the seeds; `knn_dev_sweep.png` then shows the individual seed traces.

Selection maximises development macro F1, breaking exact ties first by the smallest `k` and then by the metric order euclidean, manhattan, minkowski.

In [ ]:
k_grid = protocol["hyperparameters"]["k_grid"]
mean_curve = payload["summary"]["dev_macro_f1_mean"]

print("mean dev macro F1 across seeds")
print("metric".ljust(12) + "".join(f"{'k=' + str(k):>8}" for k in k_grid))
for metric, curve in mean_curve.items():
    print(metric.ljust(12) + "".join(f"{curve[str(k)]:>8.4f}" for k in k_grid))

## Selected configurations and held-out scores

For each seed the sweep returns exactly one configuration, and only then is the held-out split scored: accuracy, macro F1 and a confusion matrix for both the handwritten model and the scikit-learn baseline. `summary` repeats the held-out scores across seeds (mean, population standard deviation, minimum, maximum) and counts how often each configuration was selected.

In [ ]:
for run in payload["runs"]:
    selection = run["selection"]
    handwritten = run["heldout"]["handwritten"]
    reference = run["heldout"]["sklearn"]
    print(
        f"seed {run['seed']}: selected {selection['metric']} (p={selection['p']}), "
        f"k={selection['k']}; dev macro F1 {selection['dev_macro_f1']:.4f}"
    )
    print(
        f"  held-out handwritten: accuracy {handwritten['accuracy']:.4f}, "
        f"macro F1 {handwritten['macro_f1']:.4f}"
    )
    print(
        f"  held-out sklearn    : accuracy {reference['accuracy']:.4f}, "
        f"macro F1 {reference['macro_f1']:.4f}; "
        f"predictions match: {run['heldout']['predictions_match']}"
    )

summary = payload["summary"]
for model in ("handwritten", "sklearn"):
    scores = summary[model]
    print(
        f"{model:12s} macro F1 mean {scores['macro_f1']['mean']:.4f} "
        f"+/- {scores['macro_f1']['std']:.4f} across seeds "
        f"{scores['macro_f1']['values']}"
    )

## Generated figures

`knn_dev_sweep.png` shows development macro F1 and accuracy against `k`, with one colour per distance, thin lines for individual seeds and stars marking the selected points. `knn_baseline.png` compares held-out accuracy and macro F1 of the handwritten model against the scikit-learn baseline per seed and shows both confusion matrices for the first seed. Both figures are regenerated by the CLI; they are not stored in this notebook.

In [ ]:
from IPython.display import Image, display

if "figures" not in payload:
    raise KeyError(
        "the evidence file has no 'figures' entry; regenerate it with "
        "`uv run --locked python -m mlshowcase.knn` from the repository root"
    )

for role, filename in sorted(payload["figures"].items()):
    figure_path = RESULTS_JSON.parent / filename
    if not figure_path.is_file():
        raise FileNotFoundError(
            f"missing figure {figure_path.relative_to(SHOWCASE_ROOT)}; regenerate the evidence "
            "with `uv run --locked python -m mlshowcase.knn` from the repository root"
        )
    print(f"{role}: {figure_path.relative_to(SHOWCASE_ROOT)}")
    display(Image(filename=str(figure_path), width=900))

## Reproducing the evidence

```sh
uv run --locked python -m mlshowcase.knn --output-dir results --seeds 42 43 44
```

- Evidence: [`../results/knn.json`](../results/knn.json)
- Reference notes: [`../docs/knn.md`](../docs/knn.md)
- Implementation: `mlshowcase/knn.py`; tests: `tests/test_knn.py`

Scores describe three seeded splits of a small bundled dataset under the recorded protocol; they are not a benchmark claim about digit recognition in general. Rerunning the CLI overwrites the JSON and both PNG figures, and this notebook picks up the new values on the next *Run All*.